[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/llms/10-gestion-contexto.ipynb)

# Gestión de contexto en LLMs

Estrategias para manejar conversaciones largas sin superar la ventana de contexto:
ventana deslizante, compresión por resumen y recuperación selectiva.

In [ ]:
import anthropic, os, json
from dataclasses import dataclass, field
from datetime import datetime

client = anthropic.Anthropic()
MODELO = 'claude-haiku-4-5-20251001'
LIMITE_TOKENS = 50_000  # umbral antes de comprimir (el modelo soporta 200k)
print('Cliente listo')

## 1. El problema: conversaciones que crecen indefinidamente

In [ ]:
def contar_tokens_mensajes(mensajes: list[dict]) -> int:
    resp = client.messages.count_tokens(
        model=MODELO,
        messages=mensajes
    )
    return resp.input_tokens

# Simular una conversación que crece
conversacion = []
turnos = [
    ('user', 'Necesito analizar los contratos de nuestra empresa. Tenemos 200 proveedores.'),
    ('assistant', 'Puedo ayudarte. ¿Qué tipo de análisis necesitas: riesgos, vencimientos, condiciones económicas?'),
    ('user', 'Primero los vencimientos. Los contratos duran entre 1 y 5 años.'),
    ('assistant', 'Para analizar vencimientos necesito: fecha de inicio, duración y condiciones de renovación.'),
    ('user', 'La mayoría se renuevan automáticamente salvo aviso 90 días antes.'),
    ('assistant', 'Entonces el riesgo principal es el "preaviso 90 días". ¿Tienes alguno venciendo en los próximos 6 meses?'),
    ('user', 'Sí, 23 contratos. El más crítico es con Acme Corp por 500k€ anuales.'),
    ('assistant', 'Acme Corp es prioritario. ¿Quieres que prepare un calendario de alertas con 90, 60 y 30 días de antelación?'),
    ('user', 'Perfecto. Además, necesito analizar también las cláusulas de penalización.'),
]

for role, content in turnos:
    conversacion.append({'role': role, 'content': content})

tokens_usados = contar_tokens_mensajes(conversacion)
print(f'Conversación: {len(conversacion)} mensajes = {tokens_usados:,} tokens')
print(f'Coste hasta aquí: ${tokens_usados * 0.80 / 1_000_000:.5f}')
print(f'Si se extiende a 100 turnos: ~{tokens_usados * 100 / len(conversacion):,.0f} tokens')

## 2. Estrategia 1 — Ventana deslizante

In [ ]:
def ventana_deslizante(mensajes: list[dict], max_mensajes: int = 10) -> list[dict]:
    """Mantiene solo los N mensajes más recientes.
    Simple pero pierde contexto antiguo. Útil para chats de soporte."""
    if len(mensajes) <= max_mensajes:
        return mensajes
    # Conservar primer mensaje de sistema si existe
    if mensajes[0]['role'] == 'system':
        return [mensajes[0]] + mensajes[-(max_mensajes - 1):]
    return mensajes[-max_mensajes:]

# Demostración
conv_larga = conversacion * 3  # simular conversación más larga
conv_recortada = ventana_deslizante(conv_larga, max_mensajes=6)

tokens_original = contar_tokens_mensajes(conv_larga)
tokens_recortado = contar_tokens_mensajes(conv_recortada)

print(f'Original:    {len(conv_larga)} mensajes → {tokens_original:,} tokens')
print(f'Ventana (6): {len(conv_recortada)} mensajes → {tokens_recortado:,} tokens')
print(f'Reducción:   {(1 - tokens_recortado/tokens_original)*100:.1f}%')

## 3. Estrategia 2 — Compresión por resumen

In [ ]:
def resumir_conversacion(mensajes: list[dict], resumen_anterior: str = '') -> str:
    """Comprime los mensajes en un resumen que preserva los hechos clave."""
    historial_texto = '\n'.join(
        f'{m["role"].upper()}: {m["content"]}'
        for m in mensajes
    )

    contexto_previo = f'\nResumen anterior:\n{resumen_anterior}\n' if resumen_anterior else ''

    resp = client.messages.create(
        model=MODELO, max_tokens=300, temperature=0,
        messages=[{'role': 'user', 'content': f'''Resume esta conversación en un párrafo conciso.
Incluye: temas tratados, decisiones tomadas, datos específicos mencionados (nombres, cifras, fechas)
y próximos pasos acordados. Usa tiempo pasado.
{contexto_previo}
CONVERSACIÓN:
{historial_texto}'''}]
    )
    return resp.content[0].text

# Resumir los primeros 6 mensajes
mensajes_a_resumir = conversacion[:6]
resumen = resumir_conversacion(mensajes_a_resumir)
print('RESUMEN GENERADO:')
print(resumen)

tokens_resumen = contar_tokens_mensajes([{'role': 'user', 'content': resumen}])
tokens_originales = contar_tokens_mensajes(mensajes_a_resumir)
print(f'\nTokens originales: {tokens_originales:,} → Tokens resumen: {tokens_resumen:,}')
print(f'Compresión: {(1 - tokens_resumen/tokens_originales)*100:.1f}%')

## 4. Estrategia 3 — Gestor de contexto adaptativo

In [ ]:
@dataclass
class GestorContexto:
    system: str = ''
    limite_tokens: int = 50_000
    mensajes_recientes: list[dict] = field(default_factory=list)
    resumen_acumulado: str = ''
    compresiones: int = 0

    def agregar(self, role: str, content: str) -> None:
        self.mensajes_recientes.append({'role': role, 'content': content})
        self._comprimir_si_necesario()

    def _comprimir_si_necesario(self) -> None:
        mensajes_para_api = self._construir_mensajes_api()
        tokens = contar_tokens_mensajes(mensajes_para_api)
        if tokens > self.limite_tokens and len(self.mensajes_recientes) > 4:
            # Comprimir la mitad más antigua
            n_comprimir = len(self.mensajes_recientes) // 2
            a_comprimir = self.mensajes_recientes[:n_comprimir]
            self.resumen_acumulado = resumir_conversacion(a_comprimir, self.resumen_acumulado)
            self.mensajes_recientes = self.mensajes_recientes[n_comprimir:]
            self.compresiones += 1
            print(f'  [Compresión #{self.compresiones}: {n_comprimir} mensajes → resumen]')

    def _construir_mensajes_api(self) -> list[dict]:
        mensajes = []
        if self.resumen_acumulado:
            mensajes.append({
                'role': 'user',
                'content': f'[Contexto previo resumido]\n{self.resumen_acumulado}'
            })
            mensajes.append({'role': 'assistant', 'content': 'Entendido, continúo con ese contexto.'})
        mensajes.extend(self.mensajes_recientes)
        return mensajes

    def responder(self, pregunta: str) -> str:
        self.agregar('user', pregunta)
        resp = client.messages.create(
            model=MODELO, max_tokens=200, temperature=0.3,
            system=self.system,
            messages=self._construir_mensajes_api()
        )
        respuesta = resp.content[0].text
        self.agregar('assistant', respuesta)
        return respuesta

    def estado(self) -> dict:
        mensajes_api = self._construir_mensajes_api()
        return {
            'mensajes_en_memoria': len(self.mensajes_recientes),
            'compresiones': self.compresiones,
            'tiene_resumen': bool(self.resumen_acumulado),
            'tokens_actuales': contar_tokens_mensajes(mensajes_api) if mensajes_api else 0,
        }

# Demo del gestor adaptativo
gestor = GestorContexto(
    system='Eres un asistente de análisis de contratos.',
    limite_tokens=2_000  # bajo para demostrar la compresión
)

preguntas = [
    'Tenemos 200 contratos con proveedores. El más importante es con Acme Corp, 500k€/año.',
    '23 contratos vencen en los próximos 6 meses. ¿Qué prioridades recomiendas?',
    'Los contratos tienen renovación automática con 90 días de preaviso.',
    '¿Cuándo debería enviar el aviso para los contratos que vencen en septiembre?',
]

for pregunta in preguntas:
    print(f'\nUser: {pregunta}')
    resp = gestor.responder(pregunta)
    print(f'Assistant: {resp[:100]}...')
    estado = gestor.estado()
    print(f'Estado: {estado}')

## 5. Calculadora de coste por estrategia

In [ ]:
def calcular_coste_estrategia(
    turnos_totales: int,
    tokens_por_turno: int,
    modelo: str = 'haiku',
) -> dict:
    precios = {'haiku': 0.80, 'sonnet': 3.00}
    precio_mtok = precios.get(modelo, 0.80)

    # Sin gestión: cada llamada incluye todo el historial
    tokens_sin_gestion = sum((i + 1) * tokens_por_turno for i in range(turnos_totales))

    # Ventana de 10: máximo 10 * tokens_por_turno por llamada
    ventana = min(10, turnos_totales)
    tokens_ventana = sum(min(i + 1, ventana) * tokens_por_turno for i in range(turnos_totales))

    # Con resumen: ~30% del coste de los mensajes comprimidos
    tokens_resumen = tokens_sin_gestion * 0.35

    return {
        'sin_gestion': tokens_sin_gestion * precio_mtok / 1_000_000,
        'ventana_deslizante': tokens_ventana * precio_mtok / 1_000_000,
        'con_resumen': tokens_resumen * precio_mtok / 1_000_000,
    }

# Escenario: chatbot de atención al cliente
escenarios = [
    {'nombre': 'Conversación corta (20 turnos)', 'turnos': 20, 'tokens_turno': 200},
    {'nombre': 'Conversación media (50 turnos)', 'turnos': 50, 'tokens_turno': 300},
    {'nombre': 'Sesión larga (100 turnos)',      'turnos': 100, 'tokens_turno': 400},
]

print(f'{"Escenario":<35} {"Sin gestión":>12} {"Ventana":>10} {"Resumen":>10} {"Ahorro":>10}')
print('-' * 80)
for e in escenarios:
    costes = calcular_coste_estrategia(e['turnos'], e['tokens_turno'])
    ahorro = (1 - costes['con_resumen'] / costes['sin_gestion']) * 100
    print(
        f'{e["nombre"]:<35} '
        f'${costes["sin_gestion"]:>10.4f} '
        f'${costes["ventana_deslizante"]:>8.4f} '
        f'${costes["con_resumen"]:>8.4f} '
        f'{ahorro:>9.1f}%'
    )